# Color tools
Run the first cell once. The second section looks up colors; the last section writes palettes to `colors.json`, which the coffee tour web page reads.

In [1]:
import base64
import itertools
import json
import zlib
from difflib import get_close_matches

# The standard CSS color names (141 unique; "grey" spellings are handled in normalize()).
COLORS = {
    "aliceblue": "#f0f8ff", "antiquewhite": "#faebd7", "aqua": "#00ffff",
    "aquamarine": "#7fffd4", "azure": "#f0ffff", "beige": "#f5f5dc",
    "bisque": "#ffe4c4", "black": "#000000", "blanchedalmond": "#ffebcd",
    "blue": "#0000ff", "blueviolet": "#8a2be2", "brown": "#a52a2a",
    "burlywood": "#deb887", "cadetblue": "#5f9ea0", "chartreuse": "#7fff00",
    "chocolate": "#d2691e", "coral": "#ff7f50", "cornflowerblue": "#6495ed",
    "cornsilk": "#fff8dc", "crimson": "#dc143c", "cyan": "#00ffff",
    "darkblue": "#00008b", "darkcyan": "#008b8b", "darkgoldenrod": "#b8860b",
    "darkgray": "#a9a9a9", "darkgreen": "#006400", "darkkhaki": "#bdb76b",
    "darkmagenta": "#8b008b", "darkolivegreen": "#556b2f",
    "darkorange": "#ff8c00", "darkorchid": "#9932cc", "darkred": "#8b0000",
    "darksalmon": "#e9967a", "darkseagreen": "#8fbc8f",
    "darkslateblue": "#483d8b", "darkslategray": "#2f4f4f",
    "darkturquoise": "#00ced1", "darkviolet": "#9400d3",
    "deeppink": "#ff1493", "deepskyblue": "#00bfff", "dimgray": "#696969",
    "dodgerblue": "#1e90ff", "firebrick": "#b22222", "floralwhite": "#fffaf0",
    "forestgreen": "#228b22", "fuchsia": "#ff00ff", "gainsboro": "#dcdcdc",
    "ghostwhite": "#f8f8ff", "gold": "#ffd700", "goldenrod": "#daa520",
    "gray": "#808080", "green": "#008000", "greenyellow": "#adff2f",
    "honeydew": "#f0fff0", "hotpink": "#ff69b4", "indianred": "#cd5c5c",
    "indigo": "#4b0082", "ivory": "#fffff0", "khaki": "#f0e68c",
    "lavender": "#e6e6fa", "lavenderblush": "#fff0f5", "lawngreen": "#7cfc00",
    "lemonchiffon": "#fffacd", "lightblue": "#add8e6", "lightcoral": "#f08080",
    "lightcyan": "#e0ffff", "lightgoldenrodyellow": "#fafad2",
    "lightgray": "#d3d3d3", "lightgreen": "#90ee90", "lightpink": "#ffb6c1",
    "lightsalmon": "#ffa07a", "lightseagreen": "#20b2aa",
    "lightskyblue": "#87cefa", "lightslategray": "#778899",
    "lightsteelblue": "#b0c4de", "lightyellow": "#ffffe0", "lime": "#00ff00",
    "limegreen": "#32cd32", "linen": "#faf0e6", "magenta": "#ff00ff",
    "maroon": "#800000", "mediumaquamarine": "#66cdaa",
    "mediumblue": "#0000cd", "mediumorchid": "#ba55d3",
    "mediumpurple": "#9370db", "mediumseagreen": "#3cb371",
    "mediumslateblue": "#7b68ee", "mediumspringgreen": "#00fa9a",
    "mediumturquoise": "#48d1cc", "mediumvioletred": "#c71585",
    "midnightblue": "#191970", "mintcream": "#f5fffa", "mistyrose": "#ffe4e1",
    "moccasin": "#ffe4b5", "navajowhite": "#ffdead", "navy": "#000080",
    "oldlace": "#fdf5e6", "olive": "#808000", "olivedrab": "#6b8e23",
    "orange": "#ffa500", "orangered": "#ff4500", "orchid": "#da70d6",
    "palegoldenrod": "#eee8aa", "palegreen": "#98fb98",
    "paleturquoise": "#afeeee", "palevioletred": "#db7093",
    "papayawhip": "#ffefd5", "peachpuff": "#ffdab9", "peru": "#cd853f",
    "pink": "#ffc0cb", "plum": "#dda0dd", "powderblue": "#b0e0e6",
    "purple": "#800080", "rebeccapurple": "#663399", "red": "#ff0000",
    "rosybrown": "#bc8f8f", "royalblue": "#4169e1", "saddlebrown": "#8b4513",
    "salmon": "#fa8072", "sandybrown": "#f4a460", "seagreen": "#2e8b57",
    "seashell": "#fff5ee", "sienna": "#a0522d", "silver": "#c0c0c0",
    "skyblue": "#87ceeb", "slateblue": "#6a5acd", "slategray": "#708090",
    "snow": "#fffafa", "springgreen": "#00ff7f", "steelblue": "#4682b4",
    "tan": "#d2b48c", "teal": "#008080", "thistle": "#d8bfd8",
    "tomato": "#ff6347", "turquoise": "#40e0d0", "violet": "#ee82ee",
    "wheat": "#f5deb3", "white": "#ffffff", "whitesmoke": "#f5f5f5",
    "yellow": "#ffff00", "yellowgreen": "#9acd32",
}


def normalize(name: str) -> str:
    """Lowercase and drop spaces, hyphens and underscores; accept 'grey'."""
    cleaned = "".join(ch for ch in name.lower() if ch.isalnum())
    return cleaned.replace("grey", "gray")


def _wada_index() -> dict:
    """normalized Wada color name -> (display name, hex). Built on first use."""
    global _WADA_INDEX
    if _WADA_INDEX is None:
        _WADA_INDEX = {normalize(n): (n, h) for n, h, _ in WADA_COLORS}
    return _WADA_INDEX


_WADA_INDEX = None


def find_color(name: str, source: str = "any"):
    """Look a color up by name. Returns (matched_name, hex, source_label).

    source: "css"  - only the standard CSS names
            "wada" - only the color names from Sanzo Wada's book
            "any"  - CSS first, then Wada (CSS wins where both have a name,
                     e.g. "red"; use source="wada" to get Wada's version)

    Wada names tolerate small spelling differences (the book writes
    "Sulpher Yellow", so "Sulphur Yellow" still matches).
    A CMYK value also works as `name`: (22, 76, 100, 15) or "cmyk(22, 76, 100, 15)"
    (percentages, 0-100). The source label is then "cmyk".

    Raises KeyError if nothing matches.
    """
    cmyk = _parse_cmyk(name)
    if cmyk is not None:
        return name, cmyk_to_hex(*cmyk), "cmyk"
    key = normalize(name)
    if source in ("any", "css") and key in COLORS:
        return name, COLORS[key], "css"
    if source in ("any", "wada"):
        index = _wada_index()
        if key in index:
            display, hex_code = index[key]
            return display, hex_code, "wada"
        close = get_close_matches(key, index, n=1, cutoff=0.9)
        if close:
            display, hex_code = index[close[0]]
            return display, hex_code, "wada"
    raise KeyError(name)


def name_to_hex(name: str, source: str = "any") -> str:
    """Return the hex code for a color name (CSS or Wada), or raise KeyError."""
    return find_color(name, source)[1]


def hex_to_rgb(hex_code: str) -> tuple:
    h = hex_code.lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


def rgb_string(hex_code: str) -> str:
    """'#b1c9c3' -> 'rgb(177, 201, 195)'"""
    r, g, b = hex_to_rgb(hex_code)
    return f"rgb({r}, {g}, {b})"


def name_to_rgb(name: str, source: str = "any") -> str:
    """Return a color name (CSS or Wada) as a CSS string like 'rgb(177, 201, 195)'.

    Raises KeyError if the name isn't found.
    """
    return rgb_string(name_to_hex(name, source))


# ---------------------------------------------------------------------------
# CMYK support
#
# CMYK values are percentages from 0 to 100 (the numbers printed in the book).
# They are converted to sRGB the same way the Wada dataset was built:
# "U.S. Web Coated (SWOP) v2" -> sRGB, relative colorimetric with black point
# compensation. The conversion uses a small built-in lookup table (the data
# below), so no extra libraries or profile files are needed. Colors whose CMYK
# exactly matches one from Volume 1 return that dataset's hex code; any other
# CMYK is interpolated and is typically within a few points of the real
# conversion (about +/-1 on average).
# ---------------------------------------------------------------------------

_CMYK_LUT_N = 7  # grid points per ink: 0, 16.7, 33.3, 50, 66.7, 83.3, 100
_CMYK_LUT_DATA = (
    "eNoV12dwXOe52PGXRTRpUxJlyZYoURJFkaIkilXsDewkCAIkesdiC7bXs7unn7On1+29YQEsegc7RUqiZV3buravr319xzN3"
    "kps4bZLJx2QymcxkJoGfeT89n9/5zf/5fwzzr0zkGyYyytAUQ/cwkQ855v/SyX9Dx17QeoVWSVruikjvM9L/Icf+hSx/ReVL"
    "VAankp107L2I/r+J1b8Qc0+JyQI5ipKlDiq7k078L+Lbf8afPCFW8sQcQtbbyOrbVPp/4n/8M/6rR/jXWfwRTCy3kjM/J3L/"
    "A//rP2F/foj9kMFfhomnrcTaz4jCf41Rf69TSyoVV6iATLVI9M9U5j/r0d+o2oKiRGXZJ0l3ROFNWfyPWvlXSn5OzuhS0iPG"
    "mwT9DUn9qzr/vTw5I9VUsewW8rf5zA4x9u+Up99Ja9PigixMufixRr78Op/4V/mHX4gvJ4WnEr/m5BducdOv8el/K//lpfj7"
    "uvC9yD93cI9ucsuvcZk/lcnHeaKSIZgUaUmQ12LU9gTzj0XtUU4upyU6IZrjwpWo8JOY8A+F/INMpphKUvG4KRa9rGs/1pXf"
    "5afW0mOFZIWIFYaj2Utaepum/zZ3fzW9mEtM47EJgz7aoJa2qrG/z/5iJfVVNv4Aiy4NabMN6tRWJfFD9g/LqV9l4t+g0SdD"
    "2oOLysJWJfX1LD5Zx/UaHigTPQXiTI7cnGWeT8sT45JaFfwloTvPn87wm9LCV1OZ8bGkUon7itGunHYqrW5Kys8mx8dqFblc"
    "9BZyndn0qVRqU0J9Ul+q1Wakct2Tr3VkKidTpU1x/XH9+ejoQ7G04s7Pt2emTyTrm2KxRxO/qY6+FErPXLmH7enVE8nZTdH4"
    "wkM8vYyTc/jIJN48ThypkqDAzD2QUksCMctb6tydGne4woEcP3s/lVyM4zNR84TWNKoeLqsgK82sVRMLJWw6bxrP3K6mDpVS"
    "IK1Mr84l5ifRqTHjWOV2pXSoWAQpbWr1cXxuFZ1cGB6baaxMHSyMgaQ+tfJdbO45MvlouLbWWF4+WJgCyWjhO4x9jnkeYd0r"
    "2OUFfN8MAcqR/C8E9ive/YDrXmYvzbH7plhQ5HMvE8yzqPu+1rWkXppV9k6qoCDmvi0xT/OutUznQqphJrm3ngI5OfvNVOTJ"
    "mGu12rlQapgu7p3Ig6ya/Xot8njRuTrTOT/ZMFXfO14FWS3z9Qv68WPnylrn3HLD1PzesXGQ0cU/YYEf0KHv0MYX2InH2M6H"
    "OBiNCP8o+H/DDf6CbfyKOf6Q2XmfBRVO+EPM/2tt8Fu18aly/IG8c1UBJWH9h/n/Ljv4TarxSfL4WuLd5QQoSsLvJ/zfVwe/"
    "LjU+Lh5fLb67lAEFRfjdkv+Xs4MvJhsfTZxYGX93oQjyqvDbJ/5frg0+X2p8uHBiefq9hQrIa/8dDf8eg1dxJEWgQQK7R+Lv"
    "0MR/g6XfIsIyyicwLoBzLQT7c5L9L6Hsb8KpRTgeQ6J+VGvGlbdw6T9BU78Kjc2Hyzqc9yKZO2jip6j2HwIPvg8uzYZmtPC4"
    "B67eRgpvIPG/Br7/JfRiJvhQCS25w7ONyMQOOPXv/f/0XeCHaeilHHrqCq81wguvw5k/i8GvhHCNh3kesfHoDR57VcT/xHFP"
    "OXaUXR9mhGWus8x2nvkjm3zMxCqMHmFUC6NcZeSfsOIfIrWHkXKJLtB0xhRJXYnEfxxRfk8vPaBninSdpEeNdOkyndtG67+j"
    "v16jHuepFYKaG6amLlHj26j4b6nfrlLf5cjnOPnQQC5fIue2UomXKWg2EUzEQmEd7teQ8yr6ikZ8k2CmY5GYHglpkT41ck6J"
    "vKIwX8eiU7oW1VRIkXtl6awkbpaE59FyXctrSjYgp7qlxBkxtlmUv9JnJ9S6Ktf8UrlLLJwWspsE9Zn2eExZVeQFnzjdKUyc"
    "4mub+OhT7ZdjygtZeuwVVzuFxVP89CYutlgL5MpQpBC0ZcN30/CxFAISxEKVzhYpOkdZM1RLij6aoEE0Ml9W0wWFysojaak5"
    "KR6JC0DnZ0v5VD5DZFKWZPxOPHY4FgWqNFOsJ3M1PF02JwpNsdzhaBYoynRhLZFdxFIzpkS9KTZ2SK8CWZsufB3PPsFSa8b4"
    "4u3o/CF9Ash6ecEvTkOBiWB/LXStEv60BIM0UZojhSnSP072VcmrJWp/gQLJSHFW5uuSryb2VoSrRWF/ngdxrjCT4SaS3tF4"
    "Tzl6paB/ktVBTMxP19jxsqda6C7lLuczn2QyICrnphbZsRl3pd5dHLucq32SLgNdyU09YcbuuytLXYX5S9mZfeka0DTlmT/0"
    "IGBahprng6enQ7umwiBHyE+J0H3CuEQ0zxKnJ4lddQJkIvJjMbgqGBf4OzPcqTq3a5wDKU56lAyuxIbno3emtFMT2q4xDSQF"
    "8WEZWi4Mz2WbJtMnx9O7RpMgIYkPZqClumF2rKlePTlW3VXNg7gsPliDlpYMM3NN9ZmTtcld1TKIKfBvfJbv/G0vAg1PoAMP"
    "gq89hECBhH+NW15ibV9hDY+wA2v462sYyEXgv+Mt33Jtz9iGB8yBFeb1FRZk2fD3MfPXetsTtWFNObCk7FiSQZoP/zJvfpFp"
    "e5xqWE0eWEzuWIiClBj+bsL8vNb2qNKwUj6wUHpjPgWSUvi7RfNXs20PpxqW6wfma2/MZUFS/peA+5ugtx7ySzDkREK3MPgN"
    "HP6Ll3zhJ8chUgiR9jB1E6FeR8l/dmvPvErNL3OQZAuKN8L8azD3J2fpiTtf9WYYf3IEil0L6a+G5T865h656mVPLeIrWwL5"
    "q1B6e1D7g/3ZQ+f9knuR8k6b/eNXAtXtUOwPtl8/cHxbdD2hPKsm38KVwPRPoMT3lGuR8mRoH0oHhuhgQyS8lYW/w7F5Ak8R"
    "OEISgyRxkSJ/FCFfovIsJiUwIYwL/QR/nuC2UOw3SG4aScfQZAiN9WH6OVzbQohfh+tTcC0KlyGk0Itmz2GpV3DlReh+Pbyk"
    "hWcDcL0HqZ1FKq9g+vPQt/XQMy38wA8vdcOzZ5DJV7DYmuosK25B9rolf7sEnZBCQEJWJKQoopyIOQWsTcCPCzgQyGWBLwg8"
    "y3MOjmvl2C85FnDMIp/McYkIG7Oz+j1WO8YogBUWuNEsW6aZgpXJ3o2kj0aSgJHn2cUMM0tF6iORWgtdOUqXQESdY56lIw9I"
    "enmEnmump49Q44DWRnMONeMKpzxDCd+teOBALAh0pJqBlRQcTCCDMfRmFPtcx4BKVlKslGCgGDMQZW5ozGcaA2SmlIyJMT0Q"
    "1fo19bqqfKrIQOSL8ZIQzfu1TJ+auqYkP5UTQBAL8Rler/u0sV6lclUu7ZcKgJcLsfucvuxV53rk6SvS5H6xCng1OmHHa05b"
    "xdNW9J7P+3fnAyCO6mMhrBq2luDWPHI+h+zOoiBKarUIWqZHCvS9HH0uQ+9O00CLqFUNKamWvHI3I59NSR8mJaByaiULF9Pm"
    "XPJuOnE2Gf8wEQOKoJQnwoWaKVtpSRXPJIofxLNAluTyUjg/Z8pMNyfrZ+LjH8RKQJKJVZtj0dEz67o+6Tky4Xtz0gtSKL4C"
    "OeaDPdOhaxPhI2PwW+NhkCDxJdI+S3ZPktfGycOj5Fs1EsQi2IJsm5G66uLVmnC4KrxVFUCUQ+dTtulE13js6mj0cDn6Vllb"
    "RxWdq9qmyp1jhSvV/KFS/q1SEmgiOjdjnZzsHBu/Uhk7VKy+VcyuS+t4ae19br/12HnyvmvPqnvLmhtkMcc3gd5ngVsPoZOr"
    "wT3LwR8tB0GKdLzAe57gt+7jJ5fxPQv41gVi3U/Hc77nEXdrlTu5yO6ZY7fOsSDO2p/Feh7qt1a0kwvqnllt24wEYrz9aaHn"
    "Qe7WcvbkXGbPdHrblA6igv3peM/92q2l6snZysdTxW2TcaCLPzjNKx5bzuckAm5j0Hc5HPgJAv3KGlxyhLLuMOaFDX70EoT+"
    "OIx9b2EXbEzawaBuZsjLNPgjPw7SvzAm5iyxhE2HndqAW7nglbcFhJeG0RlzOT6SD9kz/a7UeU98m1/+Zmh52jgXs0wGbbU+"
    "R+W8q7DVp38z+Hxq+FHUvAJZ53rtk+ec41t9sUewaRS2yqjDi7m7cO9pIrCJhh5AgWoQEsMhDxzuQOBTKLKJxNZ8dDlA8xDt"
    "CtHtYfokTG/E6RWPXvRpnF91QnJbUDoREjei3JK7mPfkGF/G7k+2QvHjwehGRFp0zubckxHPmM1XuecvfhnIb4SVRcejrGuF"
    "9sxbvVN3/RPH/KMbwtoEa4oxIxhjNzGuJtZziPUBPjRO+6J0AKEhIx26TYcPRmDAYjWC0AgCJslhkmwkyS8oCkToUUxRMDmE"
    "SUO4eBMXDhACoNkqkpXRdBBNDqLxm1j0AKYDUijDdQmuQUhlACneQPOfo1lAyOXwsgjPB+DpfnjiOjL2GVIBhJKMGSnd4tRs"
    "naqzQXV/rHmBEo5rHlL1OZRAhww1yKGP5RAQ8ZiM4RJml/B2Eb8oEntEAghUVBQwQbAKfBvPX+C5j3gOcKwuJFA+PsJFWzn9"
    "PKt9xGqA5TW+inDlEbZwj8mdY7K7I2nAiCo3i7BTFmbiXqR2LlLdTRdARI6Uhr15y2DW2phxHE+73sm4gR6m825v1juQ9t1K"
    "Bo4noHeSEFBxOgu7U0h/Er0VR7+MYe/EMSBTZJp1Jdm+OHszyh7TmXd0BogMmYw641pvVL2hK8c05W1NAQJHJIrOWL5Hz17X"
    "0keV1NtKAvACnqg7ouM92uh1tXJUKb8tZwAn+mYMhklTy/jI+Zrt01HHT8btIA57p5xDE+6Wmud8xfdp2b+96gNR3FMPDdZC"
    "zdXwuRK8vwhvX88YjXKP0wOj9J0yfbZA78/T2/M0UBh3TRmoyHeK0tm89ElO3J4VgMy5RjP95VRTIXkmm/gkk9ie1oDEu0ar"
    "/aVKU750JlP8JJ3fnooDURh6PNTyYLhh1XxwaWTn8giYt4EkMvjI3rLmaFhyHZx375x3b5j2gjg+eD/QvAJdXIAOzgZ3zgY3"
    "1MMgSg2s4s2L+MU5/OA0sXMK3zhGAj0ysMI3L/AXZ7iDk9zOOrexygKVHViON8/HLk5Hv6jrOyf0jWUJKHz/Ur55LndxKvvF"
    "RHbneGpTSV1ffmUemLAOaw4L5Lb1ep3n/N4fhX1PDK4xk0cZ8fntgW5X8Iwn9CMIftSPjRowyYR5R/AuO3HaSWwJkPd75MqA"
    "JAyLbrPQYeVP2bktPnatK1vqS/NDCZcx1m7RT1rVLV5xpWOy2DPGDVSchmKbKXfCknnFo660rxW6F9n+GcfQRJtx9IS58opb"
    "n/H2p/wGEjKPBK0tYedR2ANI/5TTkXS7CK/X4vc3Q9CRYBBgSN0GxxwI5kJNHrTJhx324wAhxy181Mojds7o4m572ENeFsDM"
    "mDGpWeKwLTrs0Btd6iG3AsLCqKGmmirhkeKQLXfLkTnoTIOgPDq0oBhnQpb6kLV20175wl4CQSWH9XOYwYeb+gjrVdKxn3IB"
    "NpAJ2xnY6UHcPajvChb4BIcAjaShYCQYdofg7jByGUY/QVBAkUkfQ/kZZ4DpCjKXQsy+UAQQTMKjk17N4Vc7A0oDJO+FJIBz"
    "MVeJ8BTs3myHL33Rn9zrTwJMjDmncPeEzVNr91Yu+kofe/MAlQWhLygMGQVTszByWrDvkhxAhDjWGmAdw6zrDus5xfre4/yA"
    "R1k64KeDQ3SoiQ6fpOH3/oYVyRCkl6AGCeo2SZ0gqXfJdawiEUz2YPIALjXi4glcfBcX17GikawbTfejqVto4jga34nEACVQ"
    "8JgbqfYh5ZtI4UuksBNOA1IMJXst8cGOmPFyzHIwZt2RtAI1GIyNmHVbu+a4pLm/0Dw7oh4goZDmM6n+NmVdquAXcmiHEgQC"
    "EZAxo4S3ikSDSBwQiR3rWPG0XxSGBf6ewF/k+QM8t4P7G1Y+IWHg43e52AU2+jkb3cGogOF9fMnAFe+y+QtM9jMm+3okDhjB"
    "MtrTWRm4UTIcL5p2l8yvlCwgGjKXzR3Fket52/GcY3fe+UrWCVTUVHS3573Xsr4vM/4PM4EtqQCQCWMObssgV1PIsST6YRLd"
    "so6VRA9nIm2pyNVk5Fg88mE8skWPAIEZTmutSe1KXD0aUz6IqltUCfCcIZVtTWSuxNJH9dQHenKLogKeb1/uur7Qe3JuYN/s"
    "0I6FITBuBvFw28LwtTnTyWnLvinrjhkrqDqBjrbNOq5OO09MuvdOuHdMukExADSidRq6Wg+eGA/uHQvtGA+CHAoUunWSuDpO"
    "HK8Re6vEjtH164Zaj717E+KVMfF4VdhbEXZUhPXrBkjsvfH4lVrseCW2txTbsd4wCQmI3OJAR87Qy5iHHFZTq8N63OXcHPTM"
    "dZkzfVZ6yGEzue+NeL+0BTYFQjOtgVQXRPYFR4bCLUb4mBnZ5MMnm+l4G41305Z+unkocnQ4sskTqd/WY/c0tEM19Sh3BuQj"
    "g+ImNz9+s6w3F5C2rLEr3dSbPNIX3+SSx27M6E115F7N2Fm53VM83Jvf5FRL1g7J0Qu5Bgc9phu+kQN+B8C9BaNJsIwEbPZ+"
    "h+u6y/O52weQUG7Ayw37fWaozxq8Zg9/ZodBGM/0EMwA6TGQPSby6gj16QgFQnSqU470yu5BqXtYvGISPjUJIMgl23NUd8bV"
    "l+oaSlweju83xEBATLROUJ01Z2+5c6BwyZD7ZDALArLq70Cg3pHgYGvYeB627EZsIOKTXUbYY7H4bPcCznOQe3fQC8iwZHOH"
    "HF6zy3/XA531BT/0hQCBC2YUsmJGO97sxM+4iQ9cBMBoblgImPlhK3/Hzp92cB/YOYCy7FDKb0wYzPGmkegpm/6+VQeIwAxU"
    "fIbSkKnQZMmeGsm8b0kBWMSIDgfZ00MNXKeGj9LmtxgL4AMIYrCh5m7Meg2zH8GdbxHrWMFw0GkNuTvD3quw/zASeBMNgAge"
    "8oUtAbgDQq4E0UMh7M3QOlZU0M2YvUy7j7nsZw4GmDchBpAM5Iya3HqbR7vkVQ/61J/6FUBwAXve6My1ujMNntQXntRPvTGA"
    "C065vV/qbpL6z0iGfbJxm2oCMmTnhvo4423Ocpq37uPtPxbsQIBttL0n4myMuE9FvHsZ37b1suJwKwF1E8FbZPgkGf6YhLdR"
    "61hRIyjZhZE3MeokTn2MU9uIv2FlgeVORL6BSCcQaQ8ibcPWseLMoVRnOHkDTpyA43vC8W2oCmi+L9t6J9N1Id37RXrg7ewg"
    "SA0DDepN9jclDBfixgNxy9vJdRZsQEZ6oiO3ddt53XFAd70ddQHVsx5RXaq3UfGfVQKfy9DPFQhIYcBTXRJ6S0LPithnIvZz"
    "EQM8CbhIp8DdFLgzPPcZz/2M5zawHGDZDi56k9PPcPpnrPZzVt3AiIDhGifvnq+3H5roer/es3WqFxQMIBq8Nd5zrtZ/sDa0"
    "q2bYNjYMMut+IjerxrMV8xflkV1l69ayFSQ9QMZvlJxniq4DBc97Bc/WvAfEwkCirueDp3OhA9nQe9nQ1kwY6DgQI9ez1OkM"
    "9XmaejdNbU2uN0wE8My1tHQ6LX2eEt9Nilvjf2sYwLNjHS1aTwc80GscHrptNh222DZAruqdAaVtONRlMfTZGgedhwyeDT6o"
    "dMMh3XFDrd7BLv/NXuhgX2iDBy1cRoSbqL8ZG2jDb3TiX3QSG9x07oLAXeN9jXx/C3e9jTvQxm1wcpnTafZS0nsj3tcUvX5X"
    "P3BX3+AQ0ydrzMWy51qhtzF3rTnzeXN6g0NJDLWQxnaHpafTNnjJYdznHAGoO9rdj/cb7EPmDqO1weLYa3EDOKi12tAup7XP"
    "3Tbou2AIfGyAQAiV74Tg1rClE2ntRc/3o3v6MQBR0i0m1MyY25i7Xcy5HmZPDwMCrHAtHrwdNbXod9u1s53qR10q8Av85RJ0"
    "M2+8k225lz7bnvyoIwl8UsTW4nW2D7p7bnsHT/iH3/WbAeWljH1uy1C/1XTLMXLcad/pcgAiRPRbnQZ7n8l10+L50urbafMD"
    "DMO6Ao6+YM9g6MYwfMwEv2NGAEIhbZStm+ruo64P0keH6HeM9HqxwHdVa7vS1a1c65OP9klvG2QQ5sN3siOt6c7O5NXuxJGe"
    "+NuDMRAS/FDLcLD9Xrj7Itz/OWp4jRgGnM/r7h3yDrb4hy8ELJ9B1ldhK4iE3VbLgMPa7HKc87g+9XhehTyAwpwmb7/F32SD"
    "ztqD+x2hV9exIknHENZnxG+b8TMjxP4R4lUPAfCIrU/sHRQbh4XTJuETo7DdyQOMtfUkevrjjUPR0wZ931B0u0MBGG+kmtup"
    "tmt017FI3wfswGZ+CIh+A9LTivZfwQxHcdP7hHkzvY5VeAgy3gtaLodtR2DH+7Bz89+wwgY8rhaf55Lfdzjg3xUIbEaDIEL2"
    "O8LNLrjBgxzyou950M1hHNCRPitzx85edDIHncx7TnZzkAUU22vWmqzqBZt60K68a1c2QyIguVat6Zp276TWsU/v3hHvAfog"
    "UAL3hM6rYu8JcWCvNPS6YgCSGYhwS2ToMmM8zpg/Zkde50YA7wQ81kzYL5HOY6RrD+V+jXYDBgIceQcJNKDQUSz4ERZ6DQ8C"
    "CgUs3RQkLobJI2Fyd5h8FSEBQYMIczsgXICEI0Fhd1B4Ncyvlx6IsJdKjcdLzftL935WbdtU7gCpwXWXGrJtx7Kd+3I9bxZ6"
    "N2b6QcwMFPhCsu9ocnBv0vDTtGFj3ARUF5Cw87r5iD6yJ2r9acy6UXMACQICeU52H1Y8exTPG6pno+QHArIeUWd5+JAAfyTC"
    "b4jwRh4DHLWO1VmWPsjRu3l6B09v5CKAZdbf/weGfI+P"
)
_CMYK_LUT = None
_KNOWN_CMYK = None


def _check_cmyk(values, check_range: bool = True) -> tuple:
    """Validate four CMYK percentages and return them as floats."""
    try:
        nums = tuple(float(v) for v in values)
    except (TypeError, ValueError):
        nums = ()
    if len(nums) != 4 or (check_range and not all(0 <= v <= 100 for v in nums)):
        raise ValueError(
            f"CMYK needs four numbers (C, M, Y, K), each from 0 to 100; got {values!r}."
        )
    return nums


def _parse_cmyk(value):
    """Return (c, m, y, k) if `value` looks like CMYK, otherwise None.

    CMYK-like means a tuple/list such as (22, 76, 100, 15), or a string
    such as "cmyk(22, 76, 100, 15)".
    """
    if isinstance(value, (tuple, list)):
        return _check_cmyk(value, check_range=False)
    if isinstance(value, str) and value.strip().lower().startswith("cmyk("):
        body = value.strip()[5:].rstrip(") ")
        return _check_cmyk([part.strip() for part in body.split(",")], check_range=False)
    return None


def _cmyk_lut() -> bytes:
    global _CMYK_LUT
    if _CMYK_LUT is None:
        _CMYK_LUT = zlib.decompress(base64.b64decode(_CMYK_LUT_DATA))
    return _CMYK_LUT


def _known_cmyk() -> dict:
    """CMYK tuple -> hex for the Volume 1 colors, so those match exactly."""
    global _KNOWN_CMYK
    if _KNOWN_CMYK is None:
        _KNOWN_CMYK = {tuple(float(v) for v in cmyk): hex_code
                       for _, hex_code, cmyk in WADA_COLORS}
    return _KNOWN_CMYK


def _lab8_to_rgb(L8: float, a8: float, b8: float) -> tuple:
    """Lab (D50) stored as bytes -> sRGB. L8 is L*255/100; a8, b8 are offset by 128."""
    L, a, b = L8 * 100 / 255, a8 - 128, b8 - 128
    fy = (L + 16) / 116
    fx, fz = fy + a / 500, fy - b / 200

    def f(t):
        return t ** 3 if t ** 3 > 0.008856 else (t - 16 / 116) / 7.787

    X, Y, Z = 0.9642 * f(fx), f(fy), 0.8249 * f(fz)
    linear = (
        3.1338561 * X - 1.6168667 * Y - 0.4906146 * Z,
        -0.9787684 * X + 1.9161415 * Y + 0.0334540 * Z,
        0.0719453 * X - 0.2289914 * Y + 1.4052427 * Z,
    )

    def encode(u):
        u = max(0.0, min(1.0, u))
        return 255 * (12.92 * u if u <= 0.0031308 else 1.055 * u ** (1 / 2.4) - 0.055)

    return tuple(int(round(encode(u))) for u in linear)


def cmyk_to_rgb(c: float, m: float, y: float, k: float) -> tuple:
    """Convert CMYK percentages (0-100) to an (r, g, b) tuple of ints."""
    vals = _check_cmyk((c, m, y, k), check_range=False)
    known = _known_cmyk().get(vals)
    if known:
        return hex_to_rgb(known)
    vals = _check_cmyk(vals)

    n, data = _CMYK_LUT_N, _cmyk_lut()
    pos = [v / 100 * (n - 1) for v in vals]
    base = [min(int(p), n - 2) for p in pos]
    frac = [p - b for p, b in zip(pos, base)]
    lab = [0.0, 0.0, 0.0]
    for corner in itertools.product((0, 1), repeat=4):
        weight, index = 1.0, 0
        for d in range(4):
            weight *= frac[d] if corner[d] else 1 - frac[d]
            index = index * n + base[d] + corner[d]
        if weight:
            for ch in range(3):
                lab[ch] += weight * data[index * 3 + ch]
    return _lab8_to_rgb(*lab)


def cmyk_to_hex(c: float, m: float, y: float, k: float) -> str:
    """Convert CMYK percentages (0-100) to a hex code like '#ae5224'."""
    return "#{:02x}{:02x}{:02x}".format(*cmyk_to_rgb(c, m, y, k))


def cmyk_to_rgb_string(c: float, m: float, y: float, k: float) -> str:
    """Convert CMYK percentages (0-100) to 'rgb(r, g, b)'."""
    return rgb_string(cmyk_to_hex(c, m, y, k))


def convert(name: str, source: str = "any") -> str:
    """Return a printable result line, with suggestions if the name is unknown."""
    try:
        matched, hex_code, origin = find_color(name, source)
    except KeyError:
        pool = {}
        if source in ("any", "css"):
            pool.update({k: k for k in COLORS})
        if source in ("any", "wada"):
            pool.update({k: v[0] for k, v in _wada_index().items()})
        matches = get_close_matches(normalize(name), pool, n=3, cutoff=0.6)
        hint = f" Did you mean: {', '.join(pool[m] for m in matches)}?" if matches else ""
        return f"{name}: unknown color name.{hint}"
    r, g, b = hex_to_rgb(hex_code)
    label = {"wada": "Wada", "cmyk": "CMYK"}.get(origin, "CSS")
    shown = f"{name} (matched {matched})" if origin == "wada" and normalize(matched) != normalize(name) else name
    return f"{shown}: {hex_code}  (rgb({r}, {g}, {b}))  [{label}]"


def show_color(*names: str, source: str = "any"):
    """In Jupyter: display a color swatch with its hex code for each name.

    Looks in the CSS names and Sanzo Wada's color names (see find_color).
    CMYK values work too: show_color((6, 6, 29, 0), "tomato").
    """
    from IPython.display import HTML, display

    cells = []
    for name in names:
        try:
            matched, hex_code, origin = find_color(name, source)
        except KeyError:
            cells.append(f"<div style='margin:4px'>{name}: unknown color</div>")
            continue
        label = name if isinstance(name, str) else "cmyk(" + ", ".join(f"{v:g}" for v in name) + ")"
        note = (f"<br><i>{matched}</i>"
                if origin == "wada" and normalize(matched) != normalize(name) else "")
        cells.append(
            f"<div style='display:inline-block;margin:4px;text-align:center;"
            f"font-family:monospace;font-size:12px'>"
            f"<div style='width:110px;height:60px;background:{hex_code};"
            f"border:1px solid #888;border-radius:6px'></div>"
            f"{label}{note}<br>{hex_code}</div>"
        )
    display(HTML("".join(cells)))


# ---------------------------------------------------------------------------
# Sanzo Wada, "A Dictionary of Color Combinations"
#
# Volume 1 data (348 sets, 159 colors) comes from the MIT-licensed dataset
# https://github.com/mattdesl/dictionary-of-colour-combinations
# Copyright (c) 2020 Matt DesLauriers, built on the original compilation by
# Dain M. Blodorn Kim (https://github.com/dblodorn/sanzo-wada).
# Hex values are converted from the book's CMYK values (SWOP v2 -> sRGB), so
# they approximate, rather than exactly match, the printed swatches.
#
# WADA_COLORS: [name, hex, [C, M, Y, K]]
# WADA_VOL1_SETS: set number -> indexes into WADA_COLORS
# ---------------------------------------------------------------------------

WADA_COLORS = [
    ["Hermosa Pink", "#f9c1ce", [0, 30, 6, 0]],
    ["Corinthian Pink", "#f8b6ba", [0, 35, 15, 0]],
    ["Cameo Pink", "#e0b3b6", [10, 32, 19, 0]],
    ["Fawn", "#d1b0a7", [18, 31, 30, 0]],
    ["Light Brown Drab", "#b59392", [8, 30, 20, 25]],
    ["Coral Red", "#f58e84", [0, 55, 40, 0]],
    ["Fresh Color", "#f6917e", [0, 53, 45, 0]],
    ["Grenadine Pink", "#f48067", [0, 62, 58, 0]],
    ["Eosine Pink", "#f37f94", [0, 63, 23, 0]],
    ["Spinel Red", "#f27291", [0, 70, 21, 0]],
    ["Old Rose", "#d46d7a", [15, 70, 40, 0]],
    ["Eugenia Red | A", "#e2625e", [7, 76, 60, 0]],
    ["Eugenia Red | B", "#da525d", [0, 80, 50, 10]],
    ["Raw Sienna", "#bb7125", [18, 58, 100, 12]],
    ["Vinaceous Tawny", "#c56127", [17, 72, 100, 6]],
    ["Jasper Red", "#eb5324", [2, 83, 100, 0]],
    ["Spectrum Red", "#e31f26", [5, 100, 100, 0]],
    ["Red Orange", "#dd4027", [9, 90, 100, 0]],
    ["Etruscan Red", "#c55347", [16, 80, 74, 6]],
    ["Burnt Sienna", "#ae5224", [22, 76, 100, 15]],
    ["Ochre Red", "#ab544d", [18, 73, 63, 20]],
    ["Scarlet", "#cb2f43", [10, 95, 72, 7]],
    ["Carmine", "#cc1236", [0, 100, 75, 16]],
    ["Indian Lake", "#c53c69", [12, 89, 35, 9]],
    ["Rosolanc Purple", "#b73f74", [30, 90, 33, 0]],
    ["Pomegranite Purple", "#b71f57", [23, 100, 50, 6]],
    ["Hydrangea Red", "#a94151", [38, 90, 70, 0]],
    ["Brick Red", "#a84222", [22, 84, 100, 18]],
    ["Carmine Red", "#a62c37", [25, 95, 80, 16]],
    ["Pompeian Red", "#ab2439", [18, 97, 74, 19]],
    ["Red", "#a72144", [30, 100, 70, 10]],
    ["Brown", "#7c4226", [35, 74, 90, 35]],
    ["Hay's Russet", "#793327", [37, 85, 87, 35]],
    ["Vandyke Red", "#82241f", [32, 95, 95, 33]],
    ["Pansy Purple", "#7d133a", [34, 100, 60, 34]],
    ["Pale Burnt Lake", "#802626", [25, 90, 80, 40]],
    ["Violet Red", "#642d5e", [75, 100, 50, 5]],
    ["Vistoris Lake", "#6d4145", [40, 71, 55, 40]],
    ["Sulpher Yellow", "#f5ecc2", [4, 4, 28, 0]],
    ["Pale Lemon Yellow", "#ffefae", [0, 4, 38, 0]],
    ["Naples Yellow", "#fbe6a0", [2, 7, 44, 0]],
    ["Ivory Buff", "#ebd3a2", [8, 15, 40, 0]],
    ["Seashell Pink", "#fdd4bd", [0, 19, 23, 0]],
    ["Light Pinkish Cinnamon", "#fcc79b", [0, 25, 40, 0]],
    ["Pinkish Cinnamon", "#eeb480", [5, 32, 53, 0]],
    ["Cinnamon Buff", "#fdc57e", [0, 25, 57, 0]],
    ["Cream Yellow", "#fdbf68", [0, 28, 68, 0]],
    ["Golden Yellow", "#f3a257", [2, 42, 74, 0]],
    ["Vinaceous Cinnamon", "#eea78c", [4, 40, 42, 0]],
    ["Ochraceous Salmon", "#d8a37b", [15, 38, 55, 0]],
    ["Isabella Color", "#c5a56e", [15, 28, 60, 10]],
    ["Maple", "#c59f6b", [5, 26, 56, 20]],
    ["Olive Buff", "#c1c494", [16, 6, 42, 12]],
    ["Ecru", "#c2ae93", [20, 25, 40, 6]],
    ["Yellow", "#fff200", [0, 0, 100, 0]],
    ["Lemon Yellow", "#f8ed43", [5, 0, 85, 0]],
    ["Apricot Yellow", "#ffdd00", [0, 10, 100, 0]],
    ["Pyrite Yellow", "#cab356", [23, 25, 80, 0]],
    ["Olive Ocher", "#d6b43e", [18, 26, 90, 0]],
    ["Yellow Ocher", "#e2b540", [12, 28, 88, 0]],
    ["Orange Yellow", "#fcb315", [0, 33, 100, 0]],
    ["Yellow Orange", "#f99d1b", [0, 45, 100, 0]],
    ["Apricot Orange", "#f68c50", [0, 55, 75, 0]],
    ["Orange", "#f37420", [0, 68, 100, 0]],
    ["Peach Red", "#f15a30", [0, 80, 90, 0]],
    ["English Red", "#d96629", [13, 73, 100, 0]],
    ["Cinnamon Rufous", "#c27544", [20, 60, 82, 5]],
    ["Orange Rufous", "#c16b27", [18, 65, 100, 8]],
    ["Sulphine Yellow", "#c19f2c", [24, 32, 100, 4]],
    ["Khaki", "#bc892b", [24, 45, 100, 6]],
    ["Citron Yellow", "#b2b73e", [35, 17, 95, 0]],
    ["Citrine", "#b09f36", [36, 32, 100, 0]],
    ["Buffy Citrine", "#96874d", [42, 40, 82, 8]],
    ["Dark Citrine", "#8b835b", [38, 34, 67, 20]],
    ["Light Grayish Olive", "#848061", [43, 36, 62, 19]],
    ["Krongbergs Green", "#84875e", [48, 35, 70, 12]],
    ["Olive", "#837e31", [48, 38, 100, 15]],
    ["Orange Citrine", "#986f2d", [28, 48, 92, 24]],
    ["Sudan Brown", "#a36752", [25, 60, 65, 19]],
    ["Olive Green", "#6b7140", [56, 40, 85, 22]],
    ["Light Brownish Olive", "#806e4b", [42, 46, 73, 24]],
    ["Deep Grayish Olive", "#635a3a", [50, 48, 78, 37]],
    ["Pale Raw Umber", "#71502f", [46, 63, 87, 32]],
    ["Sepia", "#644b1e", [48, 60, 100, 40]],
    ["Madder Brown", "#762c19", [36, 88, 100, 38]],
    ["Mars Brown Tobacco", "#653514", [39, 76, 100, 47]],
    ["Vandyke Brown", "#4b3317", [56, 71, 97, 52]],
    ["Turquoise Green", "#b5decc", [29, 0, 24, 0]],
    ["Glaucous Green", "#b4cdc2", [30, 9, 24, 0]],
    ["Dark Greenish Glaucous", "#b7c2a9", [30, 15, 36, 0]],
    ["Yellow Green", "#afd472", [35, 0, 72, 0]],
    ["Light Green Yellow", "#c7d14f", [26, 5, 85, 0]],
    ["Night Green", "#87c540", [52, 0, 100, 0]],
    ["Olive Yellow", "#a6a159", [40, 30, 80, 0]],
    ["Artemesia Green", "#709390", [57, 28, 39, 8]],
    ["Andover Green", "#6d7e77", [60, 40, 50, 10]],
    ["Rainette Green", "#8fa071", [42, 20, 62, 10]],
    ["Chromium Green", "#719470", [50, 16, 58, 20]],
    ["Pistachio Green", "#648f7b", [64, 29, 56, 6]],
    ["Sea Green", "#00b49b", [80, 0, 51, 0]],
    ["Benzol Green", "#00978d", [100, 15, 55, 0]],
    ["Light Porcelain Green", "#00908a", [86, 22, 50, 3]],
    ["Green", "#489b6e", [75, 21, 73, 0]],
    ["Dull Viridian Green", "#009465", [90, 20, 80, 0]],
    ["Oil Green", "#819238", [53, 28, 100, 8]],
    ["Diamine Green", "#1a7444", [87, 32, 91, 18]],
    ["Cossack Green", "#437742", [76, 32, 91, 18]],
    ["Lincoln Green", "#555832", [60, 48, 86, 37]],
    ["Blackish Olive", "#42533e", [56, 32, 63, 55]],
    ["Deep Slate Olive", "#253122", [76, 60, 80, 62]],
    ["Nile Blue", "#bce4e5", [25, 0, 10, 0]],
    ["Pale King's Blue", "#a7d4e4", [33, 4, 7, 0]],
    ["Light Glaucous Blue", "#a5c8d1", [35, 10, 14, 0]],
    ["Salvia Blue", "#97acc8", [41, 25, 10, 0]],
    ["Cobalt Green", "#96d1aa", [42, 0, 42, 0]],
    ["Calamine BLue", "#78cdd0", [50, 0, 20, 0]],
    ["Venice Green", "#62c6bf", [58, 0, 30, 0]],
    ["Cerulian Blue", "#0093a5", [84, 26, 32, 0]],
    ["Peacock Blue", "#00939b", [100, 19, 43, 0]],
    ["Green Blue", "#099197", [82, 24, 40, 3]],
    ["Olympic Blue", "#5a82b3", [69, 44, 10, 0]],
    ["Blue", "#006eb8", [95, 54, 0, 0]],
    ["Antwarp Blue", "#007190", [100, 40, 30, 10]],
    ["Helvetia Blue", "#005b8d", [100, 62, 19, 10]],
    ["Dark Medici Blue", "#547076", [70, 45, 45, 15]],
    ["Dusky Green", "#004f46", [100, 30, 64, 50]],
    ["Deep Lyons Blue", "#1c4286", [100, 85, 15, 6]],
    ["Violet Blue", "#40456a", [85, 79, 38, 16]],
    ["Vandar Poel's Blue", "#064f6e", [100, 73, 43, 10]],
    ["Dark Tyrian Blue", "#12354e", [90, 66, 36, 50]],
    ["Dull Violet Black", "#1e0e3f", [95, 106, 38, 50]],
    ["Deep Indigo", "#051230", [100, 92, 52, 60]],
    ["Deep Slate Green", "#112f2c", [80, 50, 60, 70]],
    ["Grayish Lavender - A", "#b5b1d8", [28, 28, 0, 0]],
    ["Grayish Lavender - B", "#c0a9b3", [25, 33, 20, 0]],
    ["Laelia Pink", "#ca92a8", [20, 48, 18, 0]],
    ["Lilac", "#b984af", [28, 54, 8, 0]],
    ["Eupatorium Purple", "#bf5892", [25, 79, 12, 0]],
    ["Light Mauve", "#9a72aa", [43, 62, 5, 0]],
    ["Aconite Violet", "#a36aa5", [39, 68, 5, 0]],
    ["Dull Blue Violet", "#80719e", [57, 60, 17, 0]],
    ["Dark Soft Violet", "#66629c", [70, 68, 13, 0]],
    ["Blue Violet", "#6450a1", [72, 80, 0, 0]],
    ["Purple Drab", "#84565b", [38, 65, 49, 26]],
    ["Deep Violet / Plumbeous", "#70727c", [61, 52, 43, 7]],
    ["Veronia Purple", "#8c4c62", [42, 78, 46, 15]],
    ["Dark Slate Purple", "#704357", [64, 85, 60, 10]],
    ["Taupe Brown", "#7a4456", [30, 70, 35, 40]],
    ["Violet Carmine", "#713b4c", [64, 90, 70, 10]],
    ["Violet", "#4f4086", [85, 90, 18, 0]],
    ["Red Violet", "#59256a", [76, 100, 25, 15]],
    ["Cotinga Purple", "#501345", [66, 100, 42, 40]],
    ["Dusky Madder Violet", "#4e1d4c", [75, 100, 46, 30]],
    ["White", "#ffffff", [0, 0, 0, 0]],
    ["Neutral Gray", "#b6bfc1", [29, 18, 20, 0]],
    ["Mineral Gray", "#a2b0ad", [33, 18, 25, 7]],
    ["Warm Gray", "#a1a39a", [37, 28, 36, 3]],
    ["Slate Color", "#34454c", [85, 70, 62, 30]],
    ["Black", "#111314", [20, 10, 15, 100]],
]

WADA_VOL1_SETS = {
    1: [65, 117],
    2: [61, 129],
    3: [13, 39],
    4: [50, 150],
    5: [106, 128],
    6: [7, 131],
    7: [63, 88],
    8: [66, 133],
    9: [36, 140],
    10: [66, 73],
    11: [41, 155],
    12: [50, 119],
    13: [13, 145],
    14: [9, 40],
    15: [100, 133],
    16: [33, 111],
    17: [12, 99],
    18: [3, 152],
    19: [85, 92],
    20: [115, 135],
    21: [7, 99],
    22: [54, 126],
    23: [45, 138],
    24: [83, 145],
    25: [18, 110],
    26: [47, 82],
    27: [1, 157],
    28: [84, 131],
    29: [75, 113],
    30: [29, 155],
    31: [17, 39],
    32: [49, 92],
    33: [13, 157],
    34: [8, 154],
    35: [4, 28],
    36: [68, 87],
    37: [27, 150],
    38: [105, 126],
    39: [22, 123],
    40: [14, 70],
    41: [73, 115],
    42: [59, 149],
    43: [1, 139],
    44: [101, 120],
    45: [42, 55],
    46: [63, 158],
    47: [18, 134],
    48: [24, 123],
    49: [111, 121],
    50: [41, 152],
    51: [28, 121],
    52: [38, 158],
    53: [61, 152],
    54: [100, 112],
    55: [10, 153],
    56: [134, 149],
    57: [147, 157],
    58: [32, 99],
    59: [8, 71],
    60: [39, 129],
    61: [91, 151],
    62: [54, 158],
    63: [37, 117],
    64: [139, 141],
    65: [68, 115],
    66: [58, 79],
    67: [120, 129],
    68: [4, 54],
    69: [156, 158],
    70: [13, 107],
    71: [29, 49],
    72: [38, 111],
    73: [82, 96],
    74: [87, 119],
    75: [111, 127],
    76: [39, 156],
    77: [12, 128],
    78: [44, 116],
    79: [84, 119],
    80: [38, 138],
    81: [47, 156],
    82: [32, 152],
    83: [52, 127],
    84: [42, 132],
    85: [14, 122],
    86: [13, 99],
    87: [1, 70],
    88: [42, 121],
    89: [61, 127],
    90: [8, 139],
    91: [37, 67],
    92: [5, 100],
    93: [71, 112],
    94: [41, 125],
    95: [32, 130],
    96: [59, 76],
    97: [1, 18],
    98: [84, 127],
    99: [39, 117],
    100: [72, 140],
    101: [2, 126],
    102: [41, 67],
    103: [66, 152],
    104: [28, 38],
    105: [2, 97],
    106: [122, 130],
    107: [56, 74],
    108: [8, 27],
    109: [39, 108],
    110: [31, 86],
    111: [39, 90],
    112: [7, 158],
    113: [42, 86],
    114: [60, 122],
    115: [40, 64],
    116: [2, 142],
    117: [22, 158],
    118: [59, 86],
    119: [112, 129],
    120: [2, 29],
    121: [31, 49, 107],
    122: [22, 46, 100],
    123: [5, 55, 147],
    124: [35, 59, 93],
    125: [3, 117, 127],
    126: [41, 59, 126],
    127: [45, 98, 141],
    128: [1, 116, 138],
    129: [56, 69, 113],
    130: [13, 28, 149],
    131: [13, 65, 118],
    132: [38, 47, 71],
    133: [33, 71, 99],
    134: [8, 138, 150],
    135: [38, 106, 113],
    136: [21, 103, 150],
    137: [18, 45, 98],
    138: [47, 55, 116],
    139: [113, 131, 154],
    140: [47, 122, 157],
    141: [63, 90, 129],
    142: [26, 68, 113],
    143: [121, 136, 156],
    144: [24, 63, 158],
    145: [31, 70, 130],
    146: [69, 81, 105],
    147: [9, 33, 87],
    148: [58, 60, 117],
    149: [58, 63, 132],
    150: [42, 70, 88],
    151: [38, 61, 128],
    152: [18, 32, 112],
    153: [8, 60, 70],
    154: [22, 54, 121],
    155: [15, 100, 131],
    156: [58, 114, 149],
    157: [34, 58, 120],
    158: [55, 66, 92],
    159: [69, 115, 133],
    160: [68, 82, 124],
    161: [31, 44, 123],
    162: [10, 96, 136],
    163: [56, 87, 122],
    164: [17, 60, 149],
    165: [2, 9, 37],
    166: [7, 40, 132],
    167: [53, 111, 128],
    168: [55, 128, 145],
    169: [1, 39, 156],
    170: [24, 60, 150],
    171: [35, 61, 88],
    172: [66, 122, 150],
    173: [55, 84, 87],
    174: [1, 134, 147],
    175: [44, 52, 142],
    176: [0, 42, 115],
    177: [35, 72, 133],
    178: [41, 112, 119],
    179: [17, 47, 126],
    180: [45, 138, 154],
    181: [28, 149, 151],
    182: [13, 86, 131],
    183: [144, 145, 150],
    184: [9, 41, 74],
    185: [4, 18, 39],
    186: [32, 49, 121],
    187: [123, 134, 139],
    188: [96, 113, 114],
    189: [55, 109, 116],
    190: [41, 65, 158],
    191: [4, 59, 121],
    192: [46, 86, 144],
    193: [7, 40, 101],
    194: [15, 42, 120],
    195: [9, 39, 154],
    196: [70, 111, 142],
    197: [8, 141, 154],
    198: [19, 56, 102],
    199: [20, 80, 126],
    200: [28, 52, 97],
    201: [7, 76, 114],
    202: [87, 114, 157],
    203: [39, 48, 107],
    204: [24, 66, 112],
    205: [35, 48, 149],
    206: [1, 47, 66],
    207: [78, 88, 158],
    208: [38, 119, 122],
    209: [41, 61, 113],
    210: [45, 55, 107],
    211: [62, 93, 131],
    212: [29, 77, 113],
    213: [48, 56, 111],
    214: [41, 78, 149],
    215: [46, 121, 137],
    216: [15, 102, 158],
    217: [35, 49, 105],
    218: [123, 133, 144],
    219: [15, 97, 125],
    220: [25, 49, 139],
    221: [28, 154, 158],
    222: [59, 61, 67],
    223: [4, 49, 87],
    224: [9, 124, 147],
    225: [22, 125, 146],
    226: [37, 46, 149],
    227: [0, 112, 117],
    228: [28, 39, 154],
    229: [47, 109, 154],
    230: [7, 87, 114],
    231: [2, 32, 120],
    232: [22, 44, 131],
    233: [28, 72, 127],
    234: [45, 82, 111],
    235: [41, 61, 134],
    236: [69, 126, 143],
    237: [28, 84, 113],
    238: [49, 151, 156],
    239: [4, 57, 88],
    240: [6, 54, 117],
    241: [17, 39, 50, 124],
    242: [8, 19, 105, 158],
    243: [13, 41, 79, 157],
    244: [4, 14, 95, 122],
    245: [28, 104, 129, 157],
    246: [1, 27, 38, 45],
    247: [13, 56, 100, 126],
    248: [8, 69, 133, 146],
    249: [32, 53, 58, 124],
    250: [57, 64, 99, 110],
    251: [30, 54, 105, 157],
    252: [12, 13, 68, 119],
    253: [55, 62, 151, 157],
    254: [1, 38, 76, 135],
    255: [13, 57, 115, 158],
    256: [48, 63, 103, 158],
    257: [16, 60, 121, 139],
    258: [35, 44, 76, 122],
    259: [55, 119, 123, 156],
    260: [10, 48, 88, 99],
    261: [30, 39, 115, 156],
    262: [12, 41, 71, 106],
    263: [19, 44, 87, 157],
    264: [1, 17, 89, 117],
    265: [10, 56, 93, 130],
    266: [16, 41, 96, 100],
    267: [46, 61, 100, 121],
    268: [4, 13, 109, 110],
    269: [13, 35, 139, 158],
    270: [12, 38, 79, 106],
    271: [25, 114, 119, 132],
    272: [39, 63, 87, 113],
    273: [0, 34, 78, 154],
    274: [64, 73, 120, 138],
    275: [18, 53, 84, 147],
    276: [8, 42, 90, 158],
    277: [9, 24, 76, 130],
    278: [46, 58, 106, 125],
    279: [13, 48, 53, 129],
    280: [12, 107, 135, 147],
    281: [39, 100, 114, 122],
    282: [12, 51, 114, 136],
    283: [20, 35, 97, 116],
    284: [11, 56, 99, 125],
    285: [4, 19, 64, 87],
    286: [19, 60, 118, 127],
    287: [8, 57, 111, 115],
    288: [61, 83, 147, 158],
    289: [55, 91, 127, 130],
    290: [37, 39, 107, 114],
    291: [91, 99, 114, 115],
    292: [39, 44, 50, 53],
    293: [13, 87, 94, 102],
    294: [38, 46, 106, 113],
    295: [46, 54, 121, 130],
    296: [38, 49, 82, 157],
    297: [19, 61, 79, 127],
    298: [13, 55, 64, 158],
    299: [23, 48, 104, 122],
    300: [7, 46, 87, 115],
    301: [16, 41, 96, 139],
    302: [46, 53, 110, 122],
    303: [40, 64, 109, 154],
    304: [32, 46, 73, 100],
    305: [44, 56, 70, 87],
    306: [55, 100, 103, 110],
    307: [22, 133, 139, 151],
    308: [3, 21, 65, 114],
    309: [47, 62, 127, 128],
    310: [38, 44, 76, 109],
    311: [29, 46, 89, 91],
    312: [19, 61, 94, 123],
    313: [22, 54, 105, 158],
    314: [8, 32, 126, 152],
    315: [7, 38, 47, 137],
    316: [33, 103, 149, 150],
    317: [43, 53, 55, 87],
    318: [80, 108, 125, 132],
    319: [13, 56, 61, 106],
    320: [5, 38, 104, 112],
    321: [4, 38, 109, 113],
    322: [16, 27, 137, 142],
    323: [45, 70, 84, 158],
    324: [29, 120, 139, 154],
    325: [12, 40, 59, 132],
    326: [38, 64, 90, 92],
    327: [8, 13, 53, 134],
    328: [27, 62, 86, 101],
    329: [46, 134, 151, 157],
    330: [52, 110, 113, 119],
    331: [23, 130, 138, 149],
    332: [5, 21, 109, 125],
    333: [19, 55, 114, 121],
    334: [42, 76, 90, 122],
    335: [33, 61, 149, 157],
    336: [8, 32, 39, 108],
    337: [37, 135, 148, 158],
    338: [28, 60, 125, 134],
    339: [49, 65, 112, 127],
    340: [64, 99, 154, 158],
    341: [7, 106, 109, 112],
    342: [1, 46, 77, 109],
    343: [19, 41, 81, 128],
    344: [45, 126, 139, 158],
    345: [32, 110, 116, 142],
    346: [24, 87, 91, 95],
    347: [93, 99, 123, 136],
    348: [52, 106, 109, 151],
}


def _build_volume_1() -> dict:
    return {
        number: [
            {"name": WADA_COLORS[i][0], "hex": WADA_COLORS[i][1],
             "cmyk": tuple(WADA_COLORS[i][2])}
            for i in indexes
        ]
        for number, indexes in WADA_VOL1_SETS.items()
    }


# volume number -> {set number -> list of color dicts}
VOLUMES = {1: _build_volume_1()}


def add_volume(volume: int, sets: dict) -> None:
    """Register your own volume (e.g. Volume 2).

    `sets` maps a set number to a list of (name, color) pairs, where color is
    a hex code or CMYK percentages (0-100):
        add_volume(2, {1: [("Some Color", "#aabbcc"), ("Other", (6, 6, 29, 0))]})
    """
    def make(name, value):
        cmyk = _parse_cmyk(value)
        if cmyk is not None:
            return {"name": name, "hex": cmyk_to_hex(*cmyk), "cmyk": cmyk}
        return {"name": name, "hex": value.lower(), "cmyk": None}

    VOLUMES[volume] = {
        number: [make(name, value) for name, value in colors]
        for number, colors in sets.items()
    }


def get_color_set(volume: int, number: int) -> list:
    """Return the colors in a given volume and color set number.

    Each color is a dict with "name", "hex", "rgb" and "cmyk" (when known).
    Raises KeyError for an unknown volume or set number.
    """
    if volume not in VOLUMES:
        have = ", ".join(str(v) for v in sorted(VOLUMES))
        raise KeyError(
            f"Volume {volume} isn't loaded (available: {have}). "
            "Use add_volume() to supply your own data."
        )
    sets = VOLUMES[volume]
    if number not in sets:
        raise KeyError(
            f"Volume {volume} has no set {number} "
            f"(valid: {min(sets)}-{max(sets)})."
        )
    return [{**c, "rgb": hex_to_rgb(c["hex"])} for c in sets[number]]


def get_color_set_hexes(volume: int, number: int) -> list:
    """Shortcut: just the hex codes of a color set."""
    return [c["hex"] for c in get_color_set(volume, number)]


def get_color_set_rgb(volume: int, number: int) -> list:
    """Shortcut: the colors of a set as 'rgb(r, g, b)' strings."""
    return [rgb_string(c["hex"]) for c in get_color_set(volume, number)]


def show_color_set(volume: int, number: int):
    """In Jupyter: display the swatches of a color set with names and hex codes."""
    from IPython.display import HTML, display

    colors = get_color_set(volume, number)
    cells = "".join(
        f"<div style='display:inline-block;margin:4px;text-align:center;"
        f"font-family:monospace;font-size:12px'>"
        f"<div style='width:110px;height:80px;background:{c['hex']};"
        f"border:1px solid #888;border-radius:6px'></div>"
        f"{c['name']}<br>{c['hex']}</div>"
        for c in colors
    )
    display(HTML(f"<div><b>Volume {volume}, set {number}</b></div>{cells}"))


def format_color_set(volume: int, number: int) -> str:
    """Plain-text version of a color set (used by the command line)."""
    lines = [f"Volume {volume}, set {number}:"]
    for c in get_color_set(volume, number):
        r, g, b = c["rgb"]
        lines.append(f"  {c['hex']}  rgb({r}, {g}, {b})  {c['name']}")
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Writing palettes to the config file the web page reads (colors.json)
#
# File format (what index.html expects):
#   {"active": "plum",
#    "palettes": {"plum": {"background": ..., "title": ..., "shopName": ...,
#                          "text": ...}}}
# "text" is used for the page text, the tour date and the tour location.
# ---------------------------------------------------------------------------

CONFIG_PATH = "colors.json"   # change this to point at the site's colors.json
ROLES = ("background", "title", "shopName", "text")

# Which color of a Wada set fills each role, by how many colors the set has.
_DEFAULT_ROLE_INDEXES = {
    2: {"background": 0, "title": 1, "shopName": 1, "text": 1},
    3: {"background": 0, "title": 2, "shopName": 2, "text": 1},
    4: {"background": 0, "title": 3, "shopName": 2, "text": 1},
}


def resolve_color(value) -> str:
    """Turn a color into 'rgb(r, g, b)'.

    Accepts a color name ("Pyrite Yellow", "tomato"), a hex code ("#cab356"),
    an rgb() string, or CMYK percentages (0-100) as a tuple/list
    (6, 6, 29, 0) or a string "cmyk(6, 6, 29, 0)".
    """
    if _parse_cmyk(value) is not None:
        return cmyk_to_rgb_string(*_parse_cmyk(value))
    text = value.strip()
    if text.lower().startswith("rgb("):
        return text
    if text.startswith("#") and len(text) == 7:
        return rgb_string(text)
    try:
        return name_to_rgb(text)
    except KeyError:
        raise KeyError(f"Don't know the color {value!r}.") from None


def _read_config(path: str) -> dict:
    try:
        with open(path, encoding="utf-8") as f:
            config = json.load(f)
    except FileNotFoundError:
        return {"active": None, "palettes": {}}
    except json.JSONDecodeError as err:
        raise ValueError(
            f"{path} isn't valid JSON ({err}). Fix it first so nothing gets overwritten."
        ) from None
    config.setdefault("palettes", {})
    config.setdefault("active", None)
    return config


def _write_config(config: dict, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
        f.write("\n")


def write_palette(name: str, background: str, title: str, shop_name: str,
                  text: str, path: str = None, make_active: bool = True,
                  overwrite: bool = True) -> dict:
    """Save a palette to the config file the web page reads.

    Each color can be a name (CSS or Wada), a hex code, an rgb() string or
    CMYK percentages such as (6, 6, 29, 0).
    `text` is used for the page text, the tour date and the tour location.
    Other palettes already in the file are kept. With make_active=True the
    page will use this palette.

    If a palette called `name` already exists it is replaced (and the printed
    message says "Replaced"). Pass overwrite=False to refuse instead: it
    raises ValueError and leaves the file untouched.
    """
    path = path or CONFIG_PATH
    palette = {
        "background": resolve_color(background),
        "title": resolve_color(title),
        "shopName": resolve_color(shop_name),
        "text": resolve_color(text),
    }
    config = _read_config(path)
    existed = name in config["palettes"]
    if existed and not overwrite:
        raise ValueError(
            f"A palette named {name!r} already exists in {path}. "
            "Use a different name or pass overwrite=True."
        )
    config["palettes"][name] = palette
    if make_active or not config["active"]:
        config["active"] = name
    _write_config(config, path)
    verb = "Replaced" if existed else "Saved"
    print(f"{verb} palette {name!r} in {path}" + (" (active)" if config["active"] == name else ""))
    return palette


def write_palette_from_set(name: str, volume: int, number: int,
                           roles: dict = None, path: str = None,
                           make_active: bool = True,
                           overwrite: bool = True) -> dict:
    """Save a Wada color set as a palette.

    Colors are assigned to roles by position in the set (see
    _DEFAULT_ROLE_INDEXES). Override with `roles`, a dict of role -> position
    (0 is the first color), e.g. roles={"background": 2, "text": 0}.
    Only the roles you list change; the rest use the defaults.
    An existing palette with the same name is replaced unless overwrite=False.
    """
    colors = get_color_set(volume, number)
    mapping = dict(_DEFAULT_ROLE_INDEXES[len(colors)])
    for role, index in (roles or {}).items():
        if role not in ROLES:
            raise KeyError(f"Unknown role {role!r}; use one of {ROLES}.")
        mapping[role] = index
    try:
        rgb = {role: rgb_string(colors[i]["hex"]) for role, i in mapping.items()}
    except IndexError:
        raise IndexError(
            f"Volume {volume} set {number} has {len(colors)} colors "
            f"(positions 0-{len(colors) - 1})."
        ) from None
    return write_palette(name, rgb["background"], rgb["title"], rgb["shopName"],
                         rgb["text"], path=path, make_active=make_active,
                         overwrite=overwrite)


def set_active_palette(name: str, path: str = None) -> None:
    """Choose which saved palette the page uses."""
    path = path or CONFIG_PATH
    config = _read_config(path)
    if name not in config["palettes"]:
        have = ", ".join(config["palettes"]) or "none yet"
        raise KeyError(f"No palette named {name!r} in {path} (have: {have}).")
    config["active"] = name
    _write_config(config, path)
    print(f"Active palette is now {name!r}.")


def list_palettes(path: str = None) -> dict:
    """Return {"active": name, "palettes": {...}} from the config file."""
    return _read_config(path or CONFIG_PATH)


def upgrade_palettes(path: str = None, title: str = "rgb(0, 0, 0)") -> None:
    """Bring older palettes up to date: add a "title" color and drop "date".

    Palettes saved before the page had a title color only have background,
    text, shopName and date. Each of those gets `title` (default black) and
    loses `date` (the tour date now uses `text`). Palettes that already have
    a title are left alone. Edit the file afterwards to pick better titles.
    """
    path = path or CONFIG_PATH
    config = _read_config(path)
    changed = []
    for name, palette in config["palettes"].items():
        if "title" not in palette or "date" in palette:
            palette.pop("date", None)
            palette.setdefault("title", resolve_color(title))
            config["palettes"][name] = {k: palette[k] for k in ROLES if k in palette}
            changed.append(name)
    if changed:
        _write_config(config, path)
    print("Updated: " + ", ".join(changed) if changed else "Nothing to update.")


def show_palette(name: str = None, path: str = None):
    """In Jupyter: preview a saved palette (default: the active one) as the page would look."""
    from IPython.display import HTML, display

    config = _read_config(path or CONFIG_PATH)
    name = name or config["active"]
    if name not in config["palettes"]:
        raise KeyError(f"No palette named {name!r}.")
    c = config["palettes"][name]
    title = c.get("title", "rgb(0, 0, 0)")   # older palettes have no title
    display(HTML(
        f"<div style='background:{c['background']};padding:16px;border-radius:8px;"
        f"font-family:Arial,sans-serif;max-width:420px'>"
        f"<div style='font-size:1.1em;font-weight:bold;text-align:center;"
        f"margin-bottom:12px;color:{title}'>SUNDAY MORNING COFFEE TOUR</div>"
        f"<div style='color:{c['text']};margin-bottom:12px'>Come have a coffee with me.</div>"
        f"<div style='font-size:1.3em;font-weight:bold;color:{c['shopName']}'>Coffee Shop Name</div>"
        f"<div style='color:{c['text']}'>Sunday, November 22, 2026 \u2022 9:30 AM</div>"
        f"<div style='color:{c['text']}'>123 Main Street, Salt Lake City</div></div>"
        f"<div style='font-family:monospace;font-size:12px;margin-top:4px'>palette: {name}</div>"
    ))


## Look up colors (CSS names and Wada color names)

In [2]:
name_to_hex("Pyrite Yellow")   # '#cab356'
name_to_rgb("Pyrite Yellow")   # 'rgb(202, 179, 86)'


'rgb(202, 179, 86)'

In [3]:
show_color("tomato", "Pyrite Yellow", "Cinnamon Buff", "Dusky Madder Violet", "Sulphur Yellow")


## Use CMYK values instead of names
Pass CMYK percentages (0-100, as printed in the book) as `(C, M, Y, K)` anywhere a color name works. Handy when a name isn't recognized.

In [4]:
cmyk_to_hex(6, 6, 29, 0)           # '#f5ecc2'-style hex code
cmyk_to_rgb_string(6, 6, 29, 0)   # 'rgb(r, g, b)'
name_to_rgb((6, 6, 29, 0))        # same thing, via the name lookups


'rgb(241, 232, 190)'

In [5]:
show_color((6, 6, 29, 0), (23, 25, 27, 0), "Pyrite Yellow")


## Look up Sanzo Wada color sets (volume, set number)

In [6]:
get_color_set(1, 176)


[{'name': 'Hermosa Pink',
  'hex': '#f9c1ce',
  'cmyk': (0, 30, 6, 0),
  'rgb': (249, 193, 206)},
 {'name': 'Seashell Pink',
  'hex': '#fdd4bd',
  'cmyk': (0, 19, 23, 0),
  'rgb': (253, 212, 189)},
 {'name': 'Calamine BLue',
  'hex': '#78cdd0',
  'cmyk': (50, 0, 20, 0),
  'rgb': (120, 205, 208)}]

In [7]:
show_color_set(1, 176)


## Write palettes to the web page's config file
First point `CONFIG_PATH` at the `colors.json` that sits next to `index.html`.

In [8]:
CONFIG_PATH = "colors.json"   # e.g. "/path/to/coffee-site/colors.json"


In [9]:
# Any mix of color names, hex codes or rgb() strings
write_palette(
    "plum2",
    background="Dusky Madder Violet",
    title="Sulphur Yellow",
    shop_name="Pyrite Yellow",
    text="Cinnamon Buff",
)


Replaced palette 'plum2' in colors.json (active)


{'background': 'rgb(78, 29, 76)',
 'title': 'rgb(245, 236, 194)',
 'shopName': 'rgb(202, 179, 86)',
 'text': 'rgb(253, 197, 126)'}

In [10]:
# Or use a Wada color set directly (volume, set number)
write_palette_from_set("wada-1-176", 1, 176)

# Change which color of the set goes where (positions start at 0)
# write_palette_from_set("wada-1-176b", 1, 176, roles={"background": 2, "text": 0})


Replaced palette 'wada-1-176' in colors.json (active)


{'background': 'rgb(249, 193, 206)',
 'title': 'rgb(120, 205, 208)',
 'shopName': 'rgb(120, 205, 208)',
 'text': 'rgb(253, 212, 189)'}

In [11]:
show_palette()                 # preview the active palette
list_palettes()                # see everything in the file
set_active_palette("plum")     # pick which one the page uses


Active palette is now 'plum'.


In [102]:
color_sets = [['2_026',['Pyrite Yellow', 'Cinnamon Buff', 'Dusky Madder Violet', 'Sulphur Yellow']],
              ['2_032',['Deep Slate Green','Pale King\'s Blue','Orange', 'Artemisia Green']],
            #   ['test',['White', 'White', 'White', 'White']],
              ['2_042',['Lincoln Green', 'Oil Green','Orange', 'Cream Yellow']],
              ['2_044',['Vandar Poel\'s Blue', 'Mineral Gray', 'Light Glaucous Blue', (6,6,29,0)]],
              ['2_049',['Cotinga Purple',(23,25,27,0), 'Hermosa Pink', 'Hydrangea Red']],
              ['2_070',['Dull Violet Black',  (25,10,45,0),'Sulphine Yellow','Cossack Green']],
              ['2_067',['Dull Violet Black', 'Artemisia Green','Yellow Ocher', 'Orange Rufous']],
              ['2_064',['Black', 'Ivory Buff', 'Red Orange','Dark Medici Blue']],
              ['2_059',[(39,76,100,47),  'Citron Yellow','Glaucous Green', 'Light Pinkish Cinnamon']],
              ['2_053',['Dusky Green', 'Yellow green', 'Spectrum Red', 'Cossack Green']],
              ['2_199',['Cossack Green', 'White' , 'Black','English Red']],
              ['2_085',['Black', 'Pale Lemon Yellow', 'Turquoise Green', 'Corinthian Pink']],
              ['2_086',['Helvetia Blue', 'Ecru', 'Glaucous Green', (52,45,78,0)]],
              ['2_097',[(85,100,0,0), 'Ochraceous Salmon', 'Naples Yellow', 'Red Orange']],
              ['2_100',['Antwarp Blue', 'Ecru','Orange Yellow','Red Orange' ]],
              ['2_114',['Violet Blue', 'Maple', 'Ivory Buff', 'Red Orange']],
              ['2_116',['Black', 'Light Porcelian Green', 'Naples Yellow', 'Neutral Grey']],
              ['2_120',['Violet Carmine', 'Neutral Grey' , 'Yellow','Antwarp Blue']],
              ['2_124',['Black', 'Calamine Blue', 'Yellow Green', 'Pomegranate Purple']],
              ['2_125',['Pale Raw Umber', 'Artemisia Green', 'Pinkish Cinnamon','Black']],
              ['2_128',['Black','Olympic Blue', 'Pistachio Green', 'Red Orange']],
              ['2_131',['Black', 'Grenadine Pink','Yellow Ocher',  'Olympic Blue']],
              ['2_137',['Dull Veridian Green', 'White', 'Ivory Buff', 'Pompeian Red']], # 'Spectrum Red'
              ['2_147',['Yellow Ocher','White',  'Cerulean Blue', 'Black']],
              ['2_148',['Sulpher Yellow', 'Mineral Grey', 'Red Orange','Pistachio Green']],
              ['2_155',['Vandar Poel\'s Blue', 'Orange Citrine', 'Pyrite Yellow','Violet Carmine']],
              ['2_159',['Grayish Lavender', 'Seashell Pink', 'Blue Violet','Spectrum Red']],
              ['2_162',['Venice Green', 'Blackish Olive', 'Pale Lemon Yellow', 'Coral Red']],
              ['2_166',['Dark Midici Blue', 'Yellow', 'Apricot Orange', 'Green Blue']],
              ['2_168',['Sulphine Yellow', 'Vandyke Brown', 'Green Blue', 'White']],
              ['2_178',[(85, 85,90,20), 'Burnt Sienna', 'Orange', (3,12,19,0)]], #'Dull Purplish Black','Pale Cinnamon-Pink'
              ['2_177',['Cossack Green',  'Light Green Yellow', 'Light Porcelain Green','Burnt Sienna']],
              ['2_181',['Carmine', 'White', 'Orange Yellow', 'Black']],
              ['2_184',[(3,12,19,0), 'English Red', 'Vandar Poel\'s Blue', 'Black']],
              ['2_185',['Spectrum Red', 'Yellow Ocher',  'Black','Diamine Green']],
              ['2_192',['Spinel Red', 'White', 'Neutral Grey', 'Yellow Ocher']],
              ['2_224',['Mineral Grey', 'White', 'Apricot Yellow', 'Green']],
            #   ['2_000',['', '', '', '']],
            #   ['2_000',['', '', '', '']],
            #   ['2_000',['', '', '', '']],
            #   ['2_000',['', '', '', '']],
            #   ['2_000',['', '', '', '']],
            ]


for palette_name,color_set in color_sets:
    write_palette(
        palette_name,
        background=color_set[0],
        text=color_set[1],
        shop_name=color_set[2],
        title=color_set[3],
    )


Replaced palette '2_026' in colors.json (active)
Replaced palette '2_032' in colors.json (active)
Replaced palette '2_042' in colors.json (active)
Replaced palette '2_044' in colors.json (active)
Replaced palette '2_049' in colors.json (active)
Replaced palette '2_070' in colors.json (active)
Replaced palette '2_067' in colors.json (active)
Replaced palette '2_064' in colors.json (active)
Replaced palette '2_059' in colors.json (active)
Replaced palette '2_053' in colors.json (active)
Replaced palette '2_199' in colors.json (active)
Replaced palette '2_085' in colors.json (active)
Replaced palette '2_086' in colors.json (active)
Replaced palette '2_097' in colors.json (active)
Replaced palette '2_100' in colors.json (active)
Replaced palette '2_114' in colors.json (active)
Replaced palette '2_116' in colors.json (active)
Replaced palette '2_120' in colors.json (active)
Replaced palette '2_124' in colors.json (active)
Replaced palette '2_125' in colors.json (active)
Replaced palette '2_

## Update older palettes
Palettes saved before the page had a title color have a `date` key and no `title`. Run this once to add a title (black by default) and drop `date`.

In [ ]:
upgrade_palettes()
